# Kámárí · 01 · Data Gathering  (Google Colab)

Download the **free/open & free-for-research** datasets from `data/registry/datasets_free_open.yaml` into `data/raw/<dataset>/`.

**Colab setup (one time):** add these as Colab **Secrets** (🔑 left sidebar): `HF_TOKEN`, `HF_NAMESPACE`, `KAGGLE_USERNAME`, `KAGGLE_KEY`, and `KAMARI_REPO_URL` (your GitHub repo). Choose a **GPU runtime** for notebook 02.

Agreement-only datasets (CASIA-Face-Africa, RFW, BFW, AgeDB, CelebA-Spoof) are **not** auto-downloaded — request access and drop them into `data/raw/<name>/` manually. Run notebooks **01 → 02 → 03 → 04 in one session** (Colab storage is ephemeral); notebook 04 is the single upload-to-HF step.

In [ ]:
# --- Kámárí bootstrap: works on Google Colab and locally ---
import os, sys, pathlib
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    REPO_URL = os.environ.get('KAMARI_REPO_URL', '')
    try:
        from google.colab import userdata
        for _k in ['HF_TOKEN','HF_NAMESPACE','KAGGLE_USERNAME','KAGGLE_KEY','FAGE_HF_REPO','KAMARI_REPO_URL']:
            try: os.environ.setdefault(_k, userdata.get(_k))
            except Exception: pass
    except Exception: pass
    REPO_URL = os.environ.get('KAMARI_REPO_URL', REPO_URL)
    if not pathlib.Path('/content/kamari/data').exists():
        assert REPO_URL, 'Set KAMARI_REPO_URL (Colab secret) to your GitHub repo URL'
        os.system(f'git clone {REPO_URL} /content/kamari')
    os.system('pip install -q -r /content/kamari/requirements-data.txt')
    REPO = pathlib.Path('/content/kamari')
else:
    REPO = next((c for c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
                 if (c/'data'/'registry').exists()), pathlib.Path.cwd().parent)
    from dotenv import load_dotenv; load_dotenv(REPO/'.env')
sys.path.append(str(REPO))
print('REPO =', REPO, '| Colab:', IN_COLAB)

In [ ]:
import zipfile, urllib.request, yaml
RAW = REPO / 'data' / 'raw'; RAW.mkdir(parents=True, exist_ok=True)
registry = yaml.safe_load(open(REPO / 'data' / 'registry' / 'datasets_free_open.yaml'))
print('Registry version:', registry['version'])
for d in registry['datasets']:
    print(f"  {d['role']:14s} {d['access']:14s} {d['name']}")

In [ ]:
assert os.environ.get('HF_TOKEN'), 'Set HF_TOKEN (Colab secret or .env)'
have_kaggle = bool(os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))
print('HF_NAMESPACE :', os.environ.get('HF_NAMESPACE'))
print('Kaggle creds :', 'yes' if have_kaggle else 'NO (Kaggle datasets will be skipped)')

In [ ]:
def download_url(url, dest_zip, extract_to):
    extract_to = pathlib.Path(extract_to)
    if extract_to.exists() and any(extract_to.iterdir()):
        print('exists, skip:', extract_to.name); return
    extract_to.mkdir(parents=True, exist_ok=True)
    print('downloading', url); urllib.request.urlretrieve(url, dest_zip)
    with zipfile.ZipFile(dest_zip) as z: z.extractall(extract_to)
    print('extracted ->', extract_to)

def download_kaggle(slug, extract_to):
    if not have_kaggle: print('no kaggle creds, skip', slug); return
    pathlib.Path(extract_to).mkdir(parents=True, exist_ok=True)
    os.system(f'kaggle datasets download -d {slug} -p "{extract_to}" --unzip')
    print('kaggle ->', extract_to)

def download_hf(repo_id, extract_to):
    from huggingface_hub import snapshot_download
    print('hf ->', snapshot_download(repo_id, repo_type='dataset', local_dir=str(extract_to), token=os.environ['HF_TOKEN']))

In [ ]:
# UTKFace (Kaggle mirror). Non-commercial research.
download_kaggle('jangedoo/utkface-new', RAW / 'utkface')

In [ ]:
# FairFace (CC BY 4.0). If the mirror rots, fetch from https://github.com/joojs/fairface
try:
    download_url('https://drive.google.com/uc?export=download&id=1Z1RqRo0_JiavaZw2yzZG6WETdZQ8qX86',
                 RAW / 'fairface.zip', RAW / 'fairface')
except Exception as e:
    print('FairFace auto-download failed — fetch manually into data/raw/fairface/.', e)

In [ ]:
# FAGE_v2 — African public figures (HF). Set FAGE_HF_REPO secret once confirmed.
if os.environ.get('FAGE_HF_REPO'):
    download_hf(os.environ['FAGE_HF_REPO'], RAW / 'fage_v2')
else:
    print('Set FAGE_HF_REPO to fetch FAGE_v2, or drop it into data/raw/fage_v2/.')

In [ ]:
# APPA-REAL (real + apparent age)
try:
    download_url('http://158.109.8.102/AppaRealAge/appa-real-release.zip',
                 RAW / 'appa_real.zip', RAW / 'appa_real')
except Exception as e:
    print('APPA-REAL auto-download failed — fetch manually into data/raw/appa_real/.', e)

In [ ]:
for sub in sorted(RAW.glob('*')):
    if sub.is_dir():
        print(f'{sub.name:20s} {sum(1 for _ in sub.rglob("*") if _.is_file()):>8d} files')
print('\nNext: run 02_cleaning_preprocessing.ipynb (GPU runtime recommended)')